# ReefScan-ID: Stage 2 patch dataset

Patches for stage 2 of the cascade: points inside coral, labelled healthy / bleached / dead. Run in Google Colab; this notebook needs no GPU.

### Setup: Colab runtime and Google Drive

In [1]:
import os
from google.colab import drive

drive.mount("/content/drive")

base_dir = "/content/drive/MyDrive/ReefScan-ID"
raw_dir = base_dir + "/RawData"
print("Drive folder found: {}".format(os.path.isdir(base_dir)))
print("Cache folder exists: {}".format(os.path.isdir(raw_dir)))
print("CPU count: {}".format(os.cpu_count()))

Mounted at /content/drive
Drive folder found: True
Cache folder exists: True
CPU count: 2


### Setup: libraries

In [2]:
!pip install -q datasets

import datasets
import numpy as np
import pandas as pd

print("datasets version: {}".format(datasets.__version__))
print("numpy version: {}".format(np.__version__))
print("pandas version: {}".format(pd.__version__))

datasets version: 4.8.5
numpy version: 2.4.6
pandas version: 2.2.3


## Section 1: Stage 2 label map

Question: which Coralscapes classes become healthy, bleached and dead for stage 2, and how many pixels does each of the three hold in the train and validation splits?

Checks: map every mask value to a stage 2 state, print the classes in each state, count the pixels of every mask value in train and validation, then print the share of each state. The test split is left out here because it is kept for the final evaluation.

### Load the dataset

In [3]:
import time
from datasets import load_dataset

start = time.time()
ds = load_dataset("EPFL-ECEO/coralscapes", cache_dir=raw_dir)
print(ds)
print("Loading took {:.1f} seconds".format(time.time() - start))

README.md:   0%|          | 0.00/13.7k [00:00<?, ?B/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 1517
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 166
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 392
    })
})
Loading took 47.3 seconds


### Class names from the dataset card

Ids and names copied from the table on the Coralscapes dataset card (read 2026-11-12), as in notebook 02.

In [4]:
class_names = {
    1: "seagrass",
    2: "trash",
    3: "other coral dead",
    4: "other coral bleached",
    5: "sand",
    6: "other coral alive",
    7: "human",
    8: "transect tools",
    9: "fish",
    10: "algae covered substrate",
    11: "other animal",
    12: "unknown hard substrate",
    13: "background",
    14: "dark",
    15: "transect line",
    16: "massive/meandering bleached",
    17: "massive/meandering alive",
    18: "rubble",
    19: "branching bleached",
    20: "branching dead",
    21: "millepora",
    22: "branching alive",
    23: "massive/meandering dead",
    24: "clam",
    25: "acropora alive",
    26: "sea cucumber",
    27: "turbinaria",
    28: "table acropora alive",
    29: "sponge",
    30: "anemone",
    31: "pocillopora alive",
    32: "table acropora dead",
    33: "meandering bleached",
    34: "stylophora alive",
    35: "sea urchin",
    36: "meandering alive",
    37: "meandering dead",
    38: "crown of thorn",
    39: "dead clam",
}

print("Classes on the card: {}".format(len(class_names)))

Classes on the card: 39


### Stage 2 label map

In [5]:
healthy_ids = [6, 17, 22, 25, 28, 31, 34, 36]
bleached_ids = [4, 16, 19, 33]
dead_ids = [3, 20, 23, 32, 37]
excluded_ids = [21, 27]

state_names = {0: "healthy", 1: "bleached", 2: "dead"}
excluded_state = 253
not_coral_state = 255

# Values that are not coral stay at the not coral code
state_map = np.full(256, not_coral_state, dtype=np.uint8)
state_map[healthy_ids] = 0
state_map[bleached_ids] = 1
state_map[dead_ids] = 2
state_map[excluded_ids] = excluded_state

for state_id, state_name in state_names.items():
    names = [class_names[v] for v in class_names if state_map[v] == state_id]
    print("{} ({} ids): {}".format(state_name, len(names), names))
excluded = [class_names[v] for v in class_names if state_map[v] == excluded_state]
print("excluded coral ({} ids): {}".format(len(excluded), excluded))
print("Ids that are not coral: {}".format(sum(1 for v in class_names if state_map[v] == not_coral_state)))

healthy (8 ids): ['other coral alive', 'massive/meandering alive', 'branching alive', 'acropora alive', 'table acropora alive', 'pocillopora alive', 'stylophora alive', 'meandering alive']
bleached (4 ids): ['other coral bleached', 'massive/meandering bleached', 'branching bleached', 'meandering bleached']
dead (5 ids): ['other coral dead', 'branching dead', 'massive/meandering dead', 'table acropora dead', 'meandering dead']
excluded coral (2 ids): ['millepora', 'turbinaria']
Ids that are not coral: 20


### Count pixels per mask value

In [6]:
pixel_counts = {}
for split in ["train", "validation"]:
    counts = np.zeros(256, dtype=np.int64)
    masks = ds[split].select_columns(["label"])
    for row in masks:
        mask = np.array(row["label"])
        counts += np.bincount(mask.ravel(), minlength=256)
    pixel_counts[split] = counts
    print("{}: {} masks counted, {} pixels".format(split, masks.num_rows, counts.sum()))

train: 1517 masks counted, 3181379584 pixels
validation: 166 masks counted, 348127232 pixels


### Pixel share per stage 2 state

In [7]:
for split in ["train", "validation"]:
    counts = pixel_counts[split]
    total = counts.sum()
    state_pixels = {state_id: counts[state_map == state_id].sum() for state_id in state_names}
    labelled = sum(state_pixels.values())
    for state_id, state_name in state_names.items():
        print("{} {}: {:.2f}% of all pixels, {:.2f}% of the stage 2 coral pixels".format(
            split, state_name, 100 * state_pixels[state_id] / total, 100 * state_pixels[state_id] / labelled))
    excluded_pixels = counts[state_map == excluded_state].sum()
    print("{} excluded coral: {:.2f}% of all pixels".format(split, 100 * excluded_pixels / total))

train healthy: 21.19% of all pixels, 84.17% of the stage 2 coral pixels
train bleached: 0.83% of all pixels, 3.28% of the stage 2 coral pixels
train dead: 3.16% of all pixels, 12.55% of the stage 2 coral pixels
train excluded coral: 0.43% of all pixels
validation healthy: 10.67% of all pixels, 55.57% of the stage 2 coral pixels
validation bleached: 1.09% of all pixels, 5.69% of the stage 2 coral pixels
validation dead: 7.43% of all pixels, 38.73% of the stage 2 coral pixels
validation excluded coral: 0.38% of all pixels


**Observation**: The 39 classes were successfully mapped into healthy (8), bleached (4), and dead (5), with Millepora and Turbinaria excluded. Bleached coral represents only 3.28% of labelled train coral pixels, making it the minority class. Validation also has a different class distribution, with a higher proportion of dead coral (38.73% vs. 12.55% in training).

## Section 2: Sampling measurements

Question: how many candidate points does a stratified grid give per state and split, and how pure are the patches around them at each crop size?

Checks: put one random point in every grid cell of every train and validation photo, keep the points whose centre pixel is healthy, bleached or dead, and measure the purity and coral coverage of the window around each point at three crop sizes. Then count the points per state, show what a cap per state per photo keeps, and show what each purity threshold keeps. No patch is cut or saved here, and the test split is not used.

### Grid settings for the measurement

The values here are trial values for the tables below, not the final settings.

In [8]:
grid_cell = 128
crop_sizes = [64, 128, 256]
seed = 0

photo_height = 1024
photo_width = 2048
margin = max(crop_sizes) // 2
cell_rows = (photo_height - 2 * margin) // grid_cell
cell_cols = (photo_width - 2 * margin) // grid_cell

print("Grid cell: {} px, crop sizes: {}".format(grid_cell, crop_sizes))
print("Margin from the photo edge: {} px".format(margin))
print("Grid cells per photo: {} rows x {} columns = {}".format(cell_rows, cell_cols, cell_rows * cell_cols))

Grid cell: 128 px, crop sizes: [64, 128, 256]
Margin from the photo edge: 128 px
Grid cells per photo: 6 rows x 14 columns = 84


### Measure points on train and validation

In [9]:
def integral_table(flags):
    table = np.zeros((flags.shape[0] + 1, flags.shape[1] + 1), dtype=np.int64)
    table[1:, 1:] = flags.cumsum(axis=0).cumsum(axis=1)
    return table


def window_sum(table, tops, lefts, size):
    return (table[tops + size, lefts + size] - table[tops, lefts + size]
            - table[tops + size, lefts] + table[tops, lefts])


cell_count = cell_rows * cell_cols
records = []
for split in ["train", "validation"]:
    start = time.time()
    masks = ds[split].select_columns(["label"])
    for photo, row in enumerate(masks):
        state = state_map[np.array(row["label"])]

        # One random point in every grid cell
        rng = np.random.default_rng(seed + photo)
        ys = margin + np.repeat(np.arange(cell_rows), cell_cols) * grid_cell + rng.integers(0, grid_cell, cell_count)
        xs = margin + np.tile(np.arange(cell_cols), cell_rows) * grid_cell + rng.integers(0, grid_cell, cell_count)

        centre = state[ys, xs].astype(np.int64)
        keep = centre <= 2
        ys, xs, centre = ys[keep], xs[keep], centre[keep]
        if len(ys) == 0:
            continue

        tables = [integral_table(state == state_id) for state_id in state_names]
        record = {"split": split, "photo": photo, "y": ys, "x": xs, "state": centre}
        for size in crop_sizes:
            tops = ys - size // 2
            lefts = xs - size // 2
            sums = np.stack([window_sum(table, tops, lefts, size) for table in tables])
            coral = sums.sum(axis=0)
            same = sums[centre, np.arange(len(centre))]
            record["purity_{}".format(size)] = same / coral
            record["coverage_{}".format(size)] = coral / (size * size)
        records.append(pd.DataFrame(record))
    print("{}: {} photos measured in {:.1f} seconds".format(split, masks.num_rows, time.time() - start))

points = pd.concat(records, ignore_index=True)
points["state_name"] = points["state"].map(state_names)
print("Points kept: {}".format(len(points)))
print(points.head(5).to_string())

train: 1517 photos measured in 191.2 seconds
validation: 166 photos measured in 22.7 seconds
Points kept: 37428
   split  photo    y     x  state  purity_64  coverage_64  purity_128  coverage_128  purity_256  coverage_256 state_name
0  train      0  336   435      0        1.0     0.739502         1.0      0.282837         1.0      0.070709    healthy
1  train      0  360  1136      0        1.0     1.000000         1.0      0.805359         1.0      0.326981    healthy
2  train      0  341  1162      0        1.0     0.999756         1.0      0.796753         1.0      0.330383    healthy
3  train      0  394  1224      0        1.0     0.767334         1.0      0.526794         1.0      0.303482    healthy
4  train      0  887  1120      0        1.0     0.598145         1.0      0.316956         1.0      0.106125    healthy


### Points per state and split

In [10]:
print("Points per state and split:")
print(pd.crosstab(points["split"], points["state_name"]))

per_photo = points.groupby(["split", "photo", "state_name"]).size().rename("points").reset_index()
summary = per_photo.groupby(["split", "state_name"])["points"].describe(percentiles=[0.5, 0.9])
summary = summary[["count", "50%", "90%", "max"]]
summary.columns = ["photos with points", "median per photo", "90th percentile", "max per photo"]
print("Points per photo, for photos that have the state:")
print(summary.to_string())

Points per state and split:
state_name  bleached  dead  healthy
split                              
train           1137  4481    28759
validation       167  1158     1726
Points per photo, for photos that have the state:
                       photos with points  median per photo  90th percentile  max per photo
split      state_name                                                                      
train      bleached                 190.0               3.0             13.0           47.0
           dead                     796.0               3.0             13.0           72.0
           healthy                 1321.0              17.0             46.0           84.0
validation bleached                  31.0               4.0             10.0           22.0
           dead                     100.0               6.5             28.1           55.0
           healthy                  144.0              10.0             25.7           52.0


### What a cap per state per photo keeps

In [11]:
caps = [5, 10, 20, 40]

kept = {}
for cap in caps:
    capped = per_photo.assign(points=per_photo["points"].clip(upper=cap))
    kept["cap {}".format(cap)] = capped.groupby(["split", "state_name"])["points"].sum()
kept["no cap"] = per_photo.groupby(["split", "state_name"])["points"].sum()

print("Points kept with a cap per state per photo:")
print(pd.DataFrame(kept).to_string())

Points kept with a cap per state per photo:
                       cap 5  cap 10  cap 20  cap 40  no cap
split      state_name                                       
train      bleached      580     806     979    1118    1137
           dead         2541    3402    4048    4408    4481
           healthy      6109   11062   18374   26255   28759
validation bleached      104     144     165     167     167
           dead          394     632     906    1121    1158
           healthy       607    1058    1519    1714    1726


### Purity and coverage per crop size

In [12]:
for size in crop_sizes:
    purity = "purity_{}".format(size)
    coverage = "coverage_{}".format(size)
    summary = points.groupby(["split", "state_name"]).agg(
        purity_p10=(purity, lambda values: values.quantile(0.10)),
        purity_p25=(purity, lambda values: values.quantile(0.25)),
        purity_median=(purity, "median"),
        coverage_median=(coverage, "median"),
    )
    print("Crop size {} px:".format(size))
    print(summary.round(3).to_string())

Crop size 64 px:
                       purity_p10  purity_p25  purity_median  coverage_median
split      state_name                                                        
train      bleached         0.805       1.000          1.000            0.967
           dead             0.739       0.999          1.000            0.958
           healthy          1.000       1.000          1.000            0.969
validation bleached         0.542       0.690          0.949            0.884
           dead             0.794       1.000          1.000            0.987
           healthy          0.911       1.000          1.000            0.883
Crop size 128 px:
                       purity_p10  purity_p25  purity_median  coverage_median
split      state_name                                                        
train      bleached         0.616       0.837          1.000            0.865
           dead             0.565       0.832          1.000            0.815
           healthy          0

### What each purity threshold keeps

In [13]:
thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]

for size in crop_sizes:
    purity = "purity_{}".format(size)
    kept = {}
    for threshold in thresholds:
        passed = points[points[purity] >= threshold]
        kept["purity >= {}".format(threshold)] = passed.groupby(["split", "state_name"]).size()
    kept["all points"] = points.groupby(["split", "state_name"]).size()
    print("Crop size {} px, points kept:".format(size))
    print(pd.DataFrame(kept).fillna(0).astype(int).to_string())

Crop size 64 px, points kept:
                       purity >= 0.5  purity >= 0.6  purity >= 0.7  purity >= 0.8  purity >= 0.9  all points
split      state_name                                                                                       
train      bleached             1111           1094           1062           1026            965        1137
           dead                 4322           4222           4092           3928           3711        4481
           healthy             28676          28576          28395          28115          27761       28759
validation bleached              154            145            124            109             95         167
           dead                 1115           1102           1072           1040            997        1158
           healthy              1720           1699           1672           1635           1560        1726
Crop size 128 px, points kept:
                       purity >= 0.5  purity >= 0.6  purity >= 0.7 

**Observation**: The stratified grid produced 37,428 candidate points across train and validation, with healthy coral dominating both splits. Larger crops showed lower purity, particularly for bleached coral in validation (median purity dropped to 0.626 at 256 px). A purity threshold of 0.7 would retain only 69 of 167 validation bleached points at that size, highlighting the need to balance patch purity against minority-class retention.

### Points kept after the threshold and the cap together

In [14]:
cap_seed = 0
shuffled = points.sample(frac=1, random_state=cap_seed)
candidates = points.groupby(["state_name", "split"]).size()

print("Seed for the random subset under the cap: {}".format(cap_seed))
print("Candidate points per state and split:")
print(candidates.to_string())

for size in crop_sizes:
    purity = "purity_{}".format(size)
    for cap in caps:
        table = {}
        for threshold in thresholds:
            # Threshold first, then the cap on the points that remain
            passed = shuffled[shuffled[purity] >= threshold]
            order = passed.groupby(["split", "photo", "state_name"]).cumcount()
            kept = passed[order < cap].groupby(["state_name", "split"]).size()
            kept = kept.reindex(candidates.index, fill_value=0)
            table["purity >= {}".format(threshold)] = [
                "{} ({:.1f}%)".format(count, 100 * count / total) for count, total in zip(kept, candidates)]
        print("Crop size {} px, cap {} per state per photo: points kept (percent of candidates)".format(size, cap))
        print(pd.DataFrame(table, index=candidates.index).to_string())

Seed for the random subset under the cap: 0
Candidate points per state and split:
state_name  split     
bleached    train          1137
            validation      167
dead        train          4481
            validation     1158
healthy     train         28759
            validation     1726
Crop size 64 px, cap 5 per state per photo: points kept (percent of candidates)
                      purity >= 0.5 purity >= 0.6 purity >= 0.7 purity >= 0.8 purity >= 0.9
state_name split                                                                           
bleached   train        565 (49.7%)   560 (49.3%)   544 (47.8%)   532 (46.8%)   507 (44.6%)
           validation   100 (59.9%)    98 (58.7%)    84 (50.3%)    78 (46.7%)    69 (41.3%)
dead       train       2444 (54.5%)  2386 (53.2%)  2313 (51.6%)  2235 (49.9%)  2134 (47.6%)
           validation   383 (33.1%)   378 (32.6%)   373 (32.2%)   360 (31.1%)   348 (30.1%)
healthy    train       6105 (21.2%)  6098 (21.2%)  6087 (21.2%)  6069 (

**Observation**: The 37,428 candidate points showed that larger crops and stricter purity thresholds reduced patch retention, particularly for bleached coral in validation. With a 256 px crop, purity threshold of 0.7, and cap of 10 per state per photo, only 557 training and 66 validation bleached points remained (49.0% and 39.5% of candidates). Smaller crops retained more bleached patches.

## Section 3: Cut and save the patches

Question: do the patches cut with the chosen settings match the counts of the combined table in Section 2, and do the saved files read back with one manifest row per patch?

Checks: select the points with the purity threshold and then the cap, cut a window around each selected point from the train and validation photos, save the arrays and the manifests to Drive, build a manifest for the NOAA folder, then read everything back and compare. The Coralscapes test split is not read here.

### Phase 3 settings

In [15]:
import json
import shutil

final_cap = 20
final_threshold = 0.7
threshold_crop = 128
save_crops = [128, 256]

local_dir = "/content/stage2_patches"
out_dir = base_dir + "/Processed/stage2_patches"
os.makedirs(local_dir, exist_ok=True)

print("Grid cell: {} px, cap: {} per state per photo, cap seed: {}".format(grid_cell, final_cap, cap_seed))
print("Purity threshold: {} measured at the {} px crop".format(final_threshold, threshold_crop))
print("Crop sizes saved: {}".format(save_crops))
print("Output folder on Drive: {}".format(out_dir))

Grid cell: 128 px, cap: 20 per state per photo, cap seed: 0
Purity threshold: 0.7 measured at the 128 px crop
Crop sizes saved: [128, 256]
Output folder on Drive: /content/drive/MyDrive/ReefScan-ID/Processed/stage2_patches


### Select the points

In [16]:
shuffled = points.sample(frac=1, random_state=cap_seed)

# Threshold first, then the cap on the points that remain
passed = shuffled[shuffled["purity_{}".format(threshold_crop)] >= final_threshold]
order = passed.groupby(["split", "photo", "state_name"]).cumcount()
selected = passed[order < final_cap].sort_values(["split", "photo", "y", "x"]).reset_index(drop=True)

selected["source"] = "coralscapes"
selected["photo_id"] = selected["split"] + "_" + selected["photo"].astype(str).str.zfill(4)
selected["row"] = selected.groupby("split").cumcount()
selected["label"] = selected["state_name"]

print("Points selected: {}".format(len(selected)))
print(pd.crosstab(selected["label"], selected["split"]).to_string())

Points selected: 24573
split     train  validation
label                      
bleached    840         106
dead       3353         777
healthy   18060        1437


### Cut the patches

In [17]:
from numpy.lib.format import open_memmap

for split in ["train", "validation"]:
    start = time.time()
    chosen = selected[selected["split"] == split]

    # Arrays are written to the Colab disk first
    arrays = {}
    for size in save_crops:
        array_path = "{}/coralscapes_{}_{}.npy".format(local_dir, split, size)
        arrays[size] = open_memmap(array_path, mode="w+", dtype=np.uint8, shape=(len(chosen), size, size, 3))

    images = ds[split].select_columns(["image"])
    for photo, group in chosen.groupby("photo"):
        image = np.array(images[int(photo)]["image"])
        for row, y, x in zip(group["row"], group["y"], group["x"]):
            for size in save_crops:
                half = size // 2
                arrays[size][row] = image[y - half:y + half, x - half:x + half]

    for size in save_crops:
        arrays[size].flush()
        print("{} {} px: array shape {}, dtype {}".format(split, size, arrays[size].shape, arrays[size].dtype))
    print("{}: {} photos cut in {:.1f} seconds".format(split, chosen["photo"].nunique(), time.time() - start))

train 128 px: array shape (22253, 128, 128, 3), dtype uint8
train 256 px: array shape (22253, 256, 256, 3), dtype uint8
train: 1379 photos cut in 156.9 seconds
validation 128 px: array shape (2320, 128, 128, 3), dtype uint8
validation 256 px: array shape (2320, 256, 256, 3), dtype uint8
validation: 151 photos cut in 15.7 seconds


### Save patches and manifest to Drive

In [18]:
manifest_columns = ["source", "split", "photo_id", "photo", "row", "y", "x", "label", "state",
                    "purity_128", "coverage_128", "purity_256", "coverage_256"]
selected[manifest_columns].to_csv(local_dir + "/coralscapes_manifest.csv", index=False)

settings = {
    "grid_cell": grid_cell,
    "margin": margin,
    "grid_seed": seed,
    "cap_per_state_per_photo": final_cap,
    "cap_seed": cap_seed,
    "purity_threshold": final_threshold,
    "purity_threshold_crop": threshold_crop,
    "crop_sizes_saved": save_crops,
    "state_names": state_names,
    "excluded_ids": excluded_ids,
    "splits_cut": ["train", "validation"],
}
with open(local_dir + "/settings.json", "w") as handle:
    json.dump(settings, handle, indent=2)

os.makedirs(out_dir, exist_ok=True)
for name in sorted(os.listdir(local_dir)):
    start = time.time()
    shutil.copy(local_dir + "/" + name, out_dir + "/" + name)
    size_mb = os.path.getsize(out_dir + "/" + name) / 1024 ** 2
    print("{}: {:.1f} MB, copied in {:.1f} seconds".format(name, size_mb, time.time() - start))

coralscapes_manifest.csv: 2.5 MB, copied in 0.0 seconds
coralscapes_train_128.npy: 1043.1 MB, copied in 7.2 seconds
coralscapes_train_256.npy: 4172.4 MB, copied in 43.4 seconds
coralscapes_validation_128.npy: 108.8 MB, copied in 0.7 seconds
coralscapes_validation_256.npy: 435.0 MB, copied in 3.4 seconds
settings.json: 0.0 MB, copied in 0.3 seconds


### NOAA manifest

In [19]:
noaa_dir = raw_dir + "/NOAA-PIFSC-ESD-CORAL-Bleaching-Dataset"
noaa_labels = {"CORAL": "healthy", "CORAL_BL": "bleached"}
noaa_splits = {"train": "train", "val": "validation"}
image_endings = (".jpg", ".jpeg", ".png")
state_ids = {name: state_id for state_id, name in state_names.items()}

rows = []
for folder, split in noaa_splits.items():
    for class_folder, label in noaa_labels.items():
        for name in sorted(os.listdir("{}/{}/{}".format(noaa_dir, folder, class_folder))):
            if name.lower().endswith(image_endings):
                rows.append({
                    "source": "noaa",
                    "split": split,
                    "photo_id": "{}/{}/{}".format(folder, class_folder, name),
                    "label": label,
                    "state": state_ids[label],
                })

noaa_manifest = pd.DataFrame(rows)
noaa_manifest.to_csv(out_dir + "/noaa_manifest.csv", index=False)
print("NOAA rows: {}".format(len(noaa_manifest)))
print(pd.crosstab(noaa_manifest["label"], noaa_manifest["split"]).to_string())

NOAA rows: 8854
split     train  validation
label                      
bleached   2751         589
healthy    4541         973


### Read back check

In [20]:
manifest = pd.read_csv(out_dir + "/coralscapes_manifest.csv")
noaa_check = pd.read_csv(out_dir + "/noaa_manifest.csv")

saved = {}
for split in ["train", "validation"]:
    rows_in_manifest = (manifest["split"] == split).sum()
    for size in save_crops:
        saved[(split, size)] = np.load("{}/coralscapes_{}_{}.npy".format(out_dir, split, size), mmap_mode="r")
        print("{} {} px: {} patches in the file, {} rows in the manifest".format(
            split, size, saved[(split, size)].shape[0], rows_in_manifest))

both = pd.concat([manifest, noaa_check], ignore_index=True)
print("Rows per source, split and label:")
print(pd.crosstab([both["source"], both["split"]], both["label"]).to_string())
print("Splits in the Coralscapes manifest: {}".format(sorted(manifest["split"].unique())))
splits_per_photo = both.groupby(["source", "photo_id"])["split"].nunique()
print("Photo ids that appear in more than one split: {}".format((splits_per_photo > 1).sum()))

# A sample of rows is compared with the mask and with the larger patch
sample = manifest.sample(20, random_state=0)
label_matches = 0
crop_matches = 0
for item in sample.itertuples():
    mask = np.array(ds[item.split].select_columns(["label"])[int(item.photo)]["label"])
    label_matches += int(state_map[mask[item.y, item.x]] == item.state)
    small = saved[(item.split, 128)][item.row]
    large = saved[(item.split, 256)][item.row]
    crop_matches += int(np.array_equal(large[64:192, 64:192], small))
print("Centre label matches the mask: {} of {} sampled rows".format(label_matches, len(sample)))
print("128 px patch equals the centre of the 256 px patch: {} of {} sampled rows".format(crop_matches, len(sample)))

train 128 px: 22253 patches in the file, 22253 rows in the manifest
train 256 px: 22253 patches in the file, 22253 rows in the manifest
validation 128 px: 2320 patches in the file, 2320 rows in the manifest
validation 256 px: 2320 patches in the file, 2320 rows in the manifest
Rows per source, split and label:
label                   bleached  dead  healthy
source      split                              
coralscapes train            840  3353    18060
            validation       106   777     1437
noaa        train           2751     0     4541
            validation       589     0      973
Splits in the Coralscapes manifest: ['train', 'validation']
Photo ids that appear in more than one split: 0
Centre label matches the mask: 20 of 20 sampled rows
128 px patch equals the centre of the 256 px patch: 20 of 20 sampled rows


**Observation**: The selected patches matched Section 2 counts: 22,253 training and 2,320 validation patches, saved at both 128 px and 256 px. Read-back counts matched the manifest, and all 20 sampled label and crop checks passed. The NOAA manifest contained 8,854 healthy/bleached images. These checks support extraction consistency, although class imbalance remains and dead examples come only from Coralscapes.